# Exploratory Data Analysis & Client Heterogeneity (Non-IID) Analysis

**Project:** Federated Deep Learning for Heart Disease Prediction  
**Simulated Clients:**
- **Hospital 1:** Cleveland Clinic Foundation (USA)
- **Hospital 2:** Hungarian Institute of Cardiology (Budapest)
- **Hospital 3:** University Hospital Zurich/Basel (Switzerland)

---
## 1. Environment Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocessing.feature_schema import PROCESSED_FEATURE_NAMES, NUM_PROCESSED_FEATURES, CONTINUOUS_FEATURES

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

## 2. Dataset Loading & Multi-Client Schema Validation
Load each client processed partition independently from `data/processed/` and verify schema consistency without merging records across hospitals.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
CLIENT_NAMES = {
    'hospital_1': 'Hospital 1 (Cleveland)',
    'hospital_2': 'Hospital 2 (Hungarian)',
    'hospital_3': 'Hospital 3 (Switzerland)'
}

clients_data = {}
for cid, name in CLIENT_NAMES.items():
    c_dir = DATA_DIR / cid
    X_train = pd.read_csv(c_dir / 'train' / 'X_train.csv')
    y_train = pd.read_csv(c_dir / 'train' / 'y_train.csv').squeeze('columns')
    X_val = pd.read_csv(c_dir / 'validation' / 'X_val.csv')
    y_val = pd.read_csv(c_dir / 'validation' / 'y_val.csv').squeeze('columns')
    X_test = pd.read_csv(c_dir / 'test' / 'X_test.csv')
    y_test = pd.read_csv(c_dir / 'test' / 'y_test.csv').squeeze('columns')
    
    X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)
    y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)
    df_full = X_full.copy()
    df_full['target'] = y_full
    
    clients_data[cid] = {
        'name': name,
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'X_full': X_full, 'y_full': y_full,
        'df_full': df_full
    }
    print(f'Loaded {name}: {len(X_full)} samples, {X_full.shape[1]} features | Train/Val/Test: {len(X_train)}/{len(X_val)}/{len(X_test)}')

## 3. Hospital-Wise Descriptive Statistics
Inspect descriptive statistics for continuous features per client independently.

In [ ]:
for cid, name in CLIENT_NAMES.items():
    print('=' * 80)
    print(f' DESCRIPTIVE STATISTICS: {name}')
    print('=' * 80)
    stats_df = clients_data[cid]['X_full'][CONTINUOUS_FEATURES].describe().T[['mean', 'std', 'min', '50%', 'max']]
    stats_df.columns = ['Mean', 'Std', 'Min', 'Median', 'Max']
    print(stats_df.round(3))
    print()

## 4. Target Distribution & Class Balance Analysis
Examine target class distribution across clients to evaluate label skew.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for idx, (cid, name) in enumerate(CLIENT_NAMES.items()):
    ax = axes[idx]
    counts = clients_data[cid]['y_full'].value_counts().sort_index()
    n = len(clients_data[cid]['y_full'])
    bars = ax.bar(['Healthy (0)', 'Disease (1)'], [counts.get(0, 0), counts.get(1, 0)],
                  color=['#2b5c8f', '#d95f02'], edgecolor='black', width=0.5)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Patient Count')
    ax.set_ylim(0, max(counts) * 1.25)
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width()/2., h + max(counts)*0.03, f'{h} ({h/n*100:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Feature Distribution Comparison Across Clients
Compare continuous feature distributions across clients to inspect covariate shift.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
colors = ['#1f77b4', '#2ca02c', '#d62728']

for idx, feat in enumerate(CONTINUOUS_FEATURES):
    ax = axes[idx]
    plot_data = [clients_data[cid]['X_full'][feat].values for cid in CLIENT_NAMES]
    labels = [n.split(' ')[0] + '\n' + n.split(' ')[1] for n in CLIENT_NAMES.values()]
    bplot = ax.boxplot(plot_data, tick_labels=labels, patch_artist=True, medianprops=dict(color='black', linewidth=1.5))
    for patch, c in zip(bplot['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_title(f'Standardized {feat.upper()}', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis
Generate independent correlation matrices for each hospital.

In [ ]:
key_cols = CONTINUOUS_FEATURES + ['sex', 'fbs', 'exang', 'cp_4', 'restecg_0', 'slope_2', 'ca_0', 'thal_3', 'target']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for idx, (cid, name) in enumerate(CLIENT_NAMES.items()):
    ax = axes[idx]
    corr = clients_data[cid]['df_full'][key_cols].corr().fillna(0.0)
    sns.heatmap(corr, ax=ax, cmap='coolwarm', vmin=-0.8, vmax=0.8, cbar=(idx == 2), linewidths=0.5)
    ax.set_title(name, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Client Comparison: Feature-Target Correlation Drift
Examine how feature-target correlations vary across hospital sites (Concept Drift).

In [ ]:
corr_records = []
for cid, name in CLIENT_NAMES.items():
    df_full = clients_data[cid]['df_full']
    corr_with_y = df_full[PROCESSED_FEATURE_NAMES].apply(lambda x: x.corr(df_full['target']))
    for feat, val in corr_with_y.items():
        corr_records.append({'Feature': feat, 'Client': name, 'Correlation': 0.0 if np.isnan(val) else val})

top_feats = ['thalach', 'oldpeak', 'exang', 'cp_4', 'sex', 'age', 'slope_2', 'ca_0', 'thal_7', 'trestbps']
sub_df = pd.DataFrame(corr_records)
sub_df = sub_df[sub_df['Feature'].isin(top_feats)]

plt.figure(figsize=(12, 5))
sns.barplot(data=sub_df, x='Feature', y='Correlation', hue='Client', edgecolor='black')
plt.axhline(0, color='black', linestyle='--')
plt.title('Feature Correlation with Heart Disease Target Across Hospital Clients', fontweight='bold')
plt.ylabel('Correlation with Target')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 8. Non-IID Statistical Quantification
Measure Total Variation Distance (TVD) and Kolmogorov-Smirnov (KS) divergences across client distributions.

In [ ]:
# Target Total Variation Distance
p_targets = {cid: np.array([clients_data[cid]['y_full'].value_counts().get(0, 0)/len(clients_data[cid]['y_full']),
                            clients_data[cid]['y_full'].value_counts().get(1, 0)/len(clients_data[cid]['y_full'])])
             for cid in CLIENT_NAMES}

pairs = [('hospital_1', 'hospital_2'), ('hospital_1', 'hospital_3'), ('hospital_2', 'hospital_3')]
print('Label Distribution Total Variation Distance (TVD in [0, 1]):')
for c1, c2 in pairs:
    tvd = 0.5 * np.sum(np.abs(p_targets[c1] - p_targets[c2]))
    print(f'  TVD({CLIENT_NAMES[c1]} vs {CLIENT_NAMES[c2]}): {tvd:.4f}')

print('\nKolmogorov-Smirnov 2-Sample Test on Continuous Features:')
for feat in CONTINUOUS_FEATURES:
    print(f'Feature: {feat}')
    for c1, c2 in pairs:
        v1 = clients_data[c1]['X_full'][feat].values
        v2 = clients_data[c2]['X_full'][feat].values
        res = stats.ks_2samp(v1, v2)
        print(f'  {c1} vs {c2}: KS-Stat = {res.statistic:.4f}, p-val = {res.pvalue:.4e}')

## 9. Train / Validation / Test Partition Analysis
Verify stratified partition counts and disease prevalence across splits.

In [ ]:
split_records = []
for cid, name in CLIENT_NAMES.items():
    d = clients_data[cid]
    for s_name, y_s in [('Train', d['y_train']), ('Val', d['y_val']), ('Test', d['y_test'])]:
        split_records.append({
            'Client': name,
            'Split': s_name,
            'Total': len(y_s),
            'Healthy (0)': y_s.value_counts().get(0, 0),
            'Disease (1)': y_s.value_counts().get(1, 0),
            'Disease %': f'{y_s.value_counts().get(1, 0)/len(y_s)*100:.1f}%'
        })
print(pd.DataFrame(split_records).to_string(index=False))

## 10. Conclusions & Federated Learning Implications

1. **Severe Label Skew (Non-IID):** Hospital 3 exhibits 93.5% disease prevalence, Hospital 1 has 45.9%, and Hospital 2 has 35.8%. FedAvg alone will experience significant client drift without regularized or weighted federated aggregation.
2. **Covariate & Missingness Shift:** Procedural tests (`ca`, `thal`, `slope`) and unrecorded `chol` in Hospital 3 introduce feature variance differences across local nodes.
3. **Identical Schema Compatibility:** All 3 clients provide uniform (N, 25) float32 inputs ready for Federated Deep Learning.